# The Live Agentic Planner — Wired End to End
### Stage 4: Twin + Forecaster + Planner + Safety Layer, Running as One Loop

`notebooks/01_agent_overview.ipynb` covered the planner's *design* (prompt, output contract, safety checks, retry loop) with no code. This notebook wires everything into a **running control loop** and shows it actually working — the real `src/agent_planner.py` and `src/safety_layer.py` from Stage 1, driven by the real `src/digital_twin.py` from Stage 2, using a rolling forecast in place of a full forecaster training run per twin instance.

*Implemented by:* `src/live_pipeline.py :: make_live_agentic_controller()`


## 1. The Missing Piece: An Actual LLM to Call

`agent_planner.plan_with_retry()` takes any callable matching `(system_prompt, user_prompt) -> raw_text` as its `llm_call_fn`. For this notebook (and the Stage 6 comparison, which needs to run hundreds of timesteps × three arms without burning API calls or requiring a key), that callable is `src/reference_llm.py :: build_reference_llm()` — a deterministic, rule-based stand-in, **not a real LLM**.

To run this against an actual model instead, swap the controller's `llm_call_fn` for `src/reference_llm.py :: real_anthropic_llm_call`, which wires up the real Anthropic API (requires `pip install anthropic` and `ANTHROPIC_API_KEY` set). Nothing else in the pipeline changes — that's the whole point of `plan_with_retry()` taking a plain callable.


## 2. A Debugging Story Worth Keeping

The first version of the reference LLM targeted the raw forecast (`mean + 1.3·std`) with no further adjustment. It **lost to the static baseline** — a 71% QoS violation rate versus the static baseline's 5%.

The bug wasn't in the safety layer or the retry loop — both worked correctly. It was a **modelling gap**: the reference LLM's target ignored that a slice's *effective* capacity (`digital_twin.py`, Sections 2–3 of `02_digital_twin.ipynb`) is always below its *nominal* allocation once fading and contention are accounted for. It was allocating exactly enough nominal bandwidth for the forecast demand, then watching 20-30% of that allocation evaporate to fading and contention — under-provisioning URLLC by construction.

The fix was a headroom multiplier (`urllc_headroom=1.9`) that inflates the target to compensate. This is exactly the kind of failure a *real* LLM planner, reasoning over the twin's actual described behaviour in its system prompt, would be less likely to make blindly — but it's also exactly the kind of thing the **safety layer cannot catch**, because an under-provisioned-but-schema-valid, capacity-respecting plan passes both checks. It's a reminder that the safety layer guarantees the plan is *safe*, never that it's *good*.


In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
from digital_twin import DigitalTwin, SliceTrafficSpec
from reference_llm import build_reference_llm
from live_pipeline import make_live_agentic_controller
from schemas import NetworkRules

RULES = NetworkRules(total_capacity_mbps=100.0, urllc_min_guarantee_mbps=30.0, max_step_change_mbps=20.0)
SPECS = [
    SliceTrafficSpec("URLLC", base_demand_mbps=22.0, noise_std_mbps=1.5,
                      spike_probability=0.03, spike_multiplier=1.9, spike_decay=0.65),
    SliceTrafficSpec("eMBB", base_demand_mbps=45.0, noise_std_mbps=6.0,
                      spike_probability=0.03, spike_multiplier=2.2, spike_decay=0.7),
]

twin = DigitalTwin(SPECS, fading_rho=0.9, fading_min_fraction=0.8, contention_strength=0.12, seed=7)
controller = make_live_agentic_controller(build_reference_llm())  # tuned defaults, see reference_llm.py

allocations = {"URLLC": 50.0, "eMBB": 50.0}
history = []
urllc_alloc, embb_alloc, urllc_latency, urllc_demand = [], [], [], []

for t in range(150):
    allocations = controller(t, history, RULES, allocations)
    obs = twin.step(allocations)
    history.append(obs)
    urllc_alloc.append(allocations["URLLC"])
    embb_alloc.append(allocations["eMBB"])
    urllc_latency.append(obs["URLLC"].latency_ms)
    urllc_demand.append(obs["URLLC"].demand_mbps)

print("Live control loop ran for 150 timesteps -- real twin, real safety layer, reference LLM.")


## 3. Watching the Planner React

The top panel shows URLLC's allocation adapting over time (fixed for the first 10 warmup steps, then live); the bottom panel shows demand vs. the QoS threshold. Look for the allocation visibly *rising* around the same time demand spikes — this is the proactive behaviour the whole project is built around, as opposed to a fixed split or a purely reactive one.


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(urllc_alloc, color='#1f4e8c', label='URLLC allocation (Mbps)')
ax1.plot(embb_alloc, color='#7f8c8d', alpha=0.6, label='eMBB allocation (Mbps)')
ax1.set_ylabel('allocation (Mbps)')
ax1.set_title('Live agentic planner: allocation over time')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(urllc_demand, color='black', alpha=0.6, label='URLLC demand (Mbps)')
ax2.plot(urllc_latency, color='#c0392b', label='URLLC latency (ms)')
ax2.axhline(10.0, color='crimson', linestyle='--', linewidth=1, label='QoS threshold (10 ms)')
ax2.set_xlabel('timestep'); ax2.set_ylabel('demand (Mbps) / latency (ms)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

violations = sum(1 for l in urllc_latency if l > 10.0)
print(f"QoS violations: {violations}/{len(urllc_latency)} ({violations/len(urllc_latency):.1%})")


## 4. What's Implemented Where

| Concept | File | Function |
|---|---|---|
| Rolling forecast + `SliceState` assembly + planner call | `src/live_pipeline.py` | `make_live_agentic_controller()` |
| Reference (offline, rule-based) LLM | `src/reference_llm.py` | `build_reference_llm()` |
| Real Anthropic API wiring | `src/reference_llm.py` | `real_anthropic_llm_call()` |
| Prompt, retry loop, output parsing | `src/agent_planner.py` | `plan_with_retry()` (unchanged from Stage 1) |
| Two-stage safety check | `src/safety_layer.py` | `run_safety_checks()` (unchanged from Stage 1) |

This notebook adds no new safety-critical logic — it only *wires together* Stage 1 through Stage 3. Adversarial testing of the safety layer itself (malformed plans, forced retries and fallbacks) is covered next, in `05_safety_layer.ipynb`.

---
*Next: → `05_safety_layer.ipynb`*
